In [1]:
import numpy as np
from sim_library.constants import k_eff, dR, m, kb
from scipy.constants import hbar, pi
import matplotlib.pyplot as plt
from pathlib import Path
from sim_library.simulation import simulate_pulses_single_atom, simulate_pulses_p_dist, simulate_alg_cooling, simulate_alg_cooling_custom
from sim_library.sequences import RR3_gate, PulseSequence, UpPulse, DownPulse, FreeEvolution
from sim_library.plotting import plot_state_trajectories, plot_bloch, plot_hist, save_gif, plot_coolingcycles_22, fit_gaussian, fit_gaussian_custom, display_table_compare
from sim_library.data_io import load_p_dists

In [50]:
def evolve_free(dt: float, state_vec: np.ndarray, delta_D: float, delta_R: float, delta_L: float, basis: np.ndarray):
    new_state_vec = np.zeros_like(state_vec)
    for i, n in enumerate(basis):
        if n % 2 == 0:
            new_state_vec[i] = state_vec[i]*np.exp(1j*n*(delta_D+(n*delta_R))*dt)
        else:
            new_state_vec[i] = state_vec[i]*np.exp(1j*(n*(delta_D+(n*delta_R))-delta_L)*dt)
    return new_state_vec

In [ ]:
def evolve_uppulse(dt: float, state_vec: np.ndarray, omega_R: float, phi_L: float, delta_D: float, delta_R: float, delta_L: float, basis: np.ndarray):
    new_state_vec = np.zeros_like(state_vec)
    start = 0
    end = len(basis)-1
    if basis[0] % 2 == 1:
        new_state_vec[0] = state_vec[0]*np.exp(1j*(basis[0]*(delta_D+(basis[0]*delta_R))-delta_L)*dt)
        start = 1
    if basis[-1] % 2 == 0:
        new_state_vec[-1] = state_vec[-1]*np.exp(1j*(basis[-1]*(delta_D+(basis[-1]*delta_R)))*dt)
    for i in range(start,end,2):
        n = basis[i]
        delta_p = delta_D + (2*n+1)*delta_R - delta_L
        omega_eff = np.sqrt((omega_R**2)+(delta_p**2))
        P = np.exp(1j*(n*(delta_D+(n*delta_R))+(delta_p/2))*dt)
        C = np.cos(omega_eff*dt/2)-(1j*(delta_p/omega_eff)*np.sin(omega_eff*dt/2))
        C_conj = np.cos(omega_eff*dt/2)+(1j*(delta_p/omega_eff)*np.sin(omega_eff*dt/2))
        S = np.exp(1j*phi_L)*(omega_R/omega_eff)*np.sin(omega_eff*dt/2)
        S_conj = np.exp(-1j*phi_L)*(omega_R/omega_eff)*np.sin(omega_eff*dt/2)
        new_state_vec[i] = P*((C*state_vec[i])-(1j*S_conj*state_vec[i+1]))
        new_state_vec[i+1] = P*((C_conj*state_vec[i+1])-(1j*S*state_vec[i]))

    return new_state_vec


In [15]:
def evolve_downpulse(dt: float, state_vec: np.ndarray, omega_R: float, phi_L: float, delta_D: float, delta_R: float, delta_L: float, basis: np.ndarray):
    new_state_vec = np.zeros_like(state_vec)
    start = 0
    end = len(basis)-1
    if basis[0] % 2 == 0:
        new_state_vec[0] = state_vec[0]*np.exp(1j*(basis[0]*(delta_D+(basis[0]*delta_R)))*dt)
        start = 1
    if basis[-1] % 2 == 1:
        new_state_vec[-1] = state_vec[-1]*np.exp(1j*(basis[-1]*(delta_D+(basis[-1]*delta_R))-delta_L)*dt)
    for i in range(start,end,2):
        n = basis[i]
        delta_p = delta_D + (2*n+1)*delta_R - delta_L
        delta_m = delta_D + (2*n+1)*delta_R + delta_L
        omega_eff = np.sqrt((omega_R**2)+(delta_m**2))
        P = np.exp(1j*(n*(delta_D+(n*delta_R))+(delta_p/2))*dt)
        C = np.cos(omega_eff*dt/2)-(1j*(delta_m/omega_eff)*np.sin(omega_eff*dt/2))
        C_conj = np.cos(omega_eff*dt/2)+(1j*(delta_m/omega_eff)*np.sin(omega_eff*dt/2))
        S = np.exp(-1j*phi_L)*(omega_R/omega_eff)*np.sin(omega_eff*dt/2)
        S_conj = np.exp(1j*phi_L)*(omega_R/omega_eff)*np.sin(omega_eff*dt/2)
        new_state_vec[i] = P*((C*state_vec[i])-(1j*S_conj*state_vec[i+1]))
        new_state_vec[i+1] = P*((C_conj*state_vec[i+1])-(1j*S*state_vec[i]))

    return new_state_vec


In [63]:

# Rabi frequency in 2*pi*Hz units
rabi_freq = 2*pi*1e5
rabi_time = 2*pi/rabi_freq # 2pi pulse time

# Set total resolution of time steps
n_steps = 100
duration = rabi_time
dt = duration/(n_steps-1) # n_steps includes initial state

# Define basis: p in units of hbar*k (even p = ground state)
p_min = -2
p_max = 2
basis = np.arange(p_min, p_max + 1)

# Define initial state vector
initial_state = np.zeros(shape=len(basis), dtype=np.complex128)
initial_state[2] = 1
initial_state[3] = 1
initial_state = initial_state/ np.linalg.norm(initial_state) # Normalise

# Set doppler shift
atom_veloc = 0 # Stationary

wave_func = np.zeros((n_steps, len(basis)), dtype=np.complex128)
wave_func[0,:] = initial_state

for i in range(n_steps-1):
    # wave_func[i+1,:] = evolve_uppulse(dt=dt, state_vec=wave_func[i,:], omega_R=rabi_freq, phi_L=pi/2, delta_D=k_eff*atom_veloc, delta_R=dR, delta_L=-5*dR, basis=basis)
    wave_func[i+1,:] = evolve_free(dt=dt, state_vec=wave_func[i,:], delta_D=k_eff*atom_veloc, delta_R=dR, delta_L=-5*dR, basis=basis)

plot_bloch(wave_func=wave_func[:,2:4], show=False)
print(wave_func[:,2:4])


[[ 0.70710678+0.j          0.70710678+0.j        ]
 [ 0.70710678+0.j          0.70588515+0.04154701j]
 [ 0.70710678+0.j          0.70222448+0.08295047j]
 [ 0.70710678+0.j          0.69613742+0.12406731j]
 [ 0.70710678+0.j          0.687645  +0.16475545j]
 [ 0.70710678+0.j          0.67677656+0.20487432j]
 [ 0.70710678+0.j          0.66356966+0.24428529j]
 [ 0.70710678+0.j          0.64806994+0.28285218j]
 [ 0.70710678+0.j          0.63033094+0.32044174j]
 [ 0.70710678+0.j          0.61041396+0.35692407j]
 [ 0.70710678+0.j          0.58838783+0.39217313j]
 [ 0.70710678+0.j          0.56432864+0.42606711j]
 [ 0.70710678+0.j          0.53831953+0.45848891j]
 [ 0.70710678+0.j          0.51045037+0.48932649j]
 [ 0.70710678+0.j          0.48081746+0.51847331j]
 [ 0.70710678+0.j          0.44952317+0.54582865j]
 [ 0.70710678+0.j          0.41667566+0.57129799j]
 [ 0.70710678+0.j          0.38238841+0.59479333j]
 [ 0.70710678+0.j          0.34677989+0.61623349j]
 [ 0.70710678+0.j          0.30

In [64]:

# Rabi frequency in 2*pi*Hz units
rabi_freq = 2*pi*1e5
rabi_time = 2*pi/rabi_freq # 2pi pulse time

# Set total resolution of time steps
n_steps = 100
duration = rabi_time
dt = duration/(n_steps-1) # n_steps includes initial state

# Define basis: p in units of hbar*k (even p = ground state)
p_min = -2
p_max = 2
basis = np.arange(p_min, p_max + 1)

# Define initial state vector
initial_state = np.zeros(shape=len(basis), dtype=np.complex128)
initial_state[2] = 1
initial_state[3] = 1
initial_state = initial_state/ np.linalg.norm(initial_state) # Normalise

# Set doppler shift
atom_veloc = 0 # Stationary

wave_func = np.zeros((n_steps+1, len(basis)), dtype=np.complex128)
wave_func[0,:] = initial_state

pulse_seq = PulseSequence()
# pulse1 = UpPulse(laser_det=np.full(n_steps-1, -5*dR), phase=np.full(n_steps-1, pi/2), rabi_freq=np.full(n_steps-1, rabi_freq),duration=duration)
pulse1 = FreeEvolution(laser_det=np.full(n_steps-1, -5*dR), duration=duration)
pulse_seq.add_pulses(pulse1)

wave_func2, times = simulate_pulses_single_atom(pulse_seq=pulse_seq, basis=basis, initial_state=initial_state, d_shift=k_eff*atom_veloc)

plot_bloch(wave_func=wave_func2[:,2:4], show=False)

print(wave_func2[:,2:4])


[[ 0.70710678+0.j          0.70710678+0.j        ]
 [ 0.70710678+0.j          0.70588515+0.04154701j]
 [ 0.70710678+0.j          0.70222448+0.08295047j]
 [ 0.70710678+0.j          0.69613742+0.12406731j]
 [ 0.70710678+0.j          0.687645  +0.16475545j]
 [ 0.70710678+0.j          0.67677656+0.20487432j]
 [ 0.70710678+0.j          0.66356966+0.24428529j]
 [ 0.70710678+0.j          0.64806994+0.28285218j]
 [ 0.70710678+0.j          0.63033094+0.32044174j]
 [ 0.70710678+0.j          0.61041396+0.35692407j]
 [ 0.70710678+0.j          0.58838783+0.39217313j]
 [ 0.70710678+0.j          0.56432864+0.42606711j]
 [ 0.70710678+0.j          0.53831953+0.45848891j]
 [ 0.70710678+0.j          0.51045037+0.48932649j]
 [ 0.70710678+0.j          0.48081746+0.51847331j]
 [ 0.70710678+0.j          0.44952317+0.54582865j]
 [ 0.70710678+0.j          0.41667566+0.57129799j]
 [ 0.70710678+0.j          0.38238841+0.59479333j]
 [ 0.70710678+0.j          0.34677989+0.61623349j]
 [ 0.70710678+0.j          0.30